# EN Llama 3 Baseline and QLoRA

Main thesis notebook for the English branch. It uses `meta-llama/Meta-Llama-3-8B-Instruct` for both baseline generation and QLoRA adaptation.

## 1. Repository setup

This notebook can run directly in Colab. If the repository files are not already present, it clones the public GitHub repository and switches the runtime working directory to the project root.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/chipalgash/llama_test.git"
REPO_DIR = Path("/content/llama_test")

if not Path("src").exists():
    if not REPO_DIR.exists():
        print(f"Cloning {REPO_URL} -> {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print("Already in project root; using current working directory.")

print("Current directory:", Path.cwd())
assert Path("src").exists(), "src/ not found. Check repository clone or %cd to project root."
assert Path("data/processed/en/eval_payload.jsonl").exists(), "EN eval payload is missing."
assert Path("data/processed/ru/eval_payload.jsonl").exists(), "RU eval payload is missing."
print("Repository is ready.")

## 2. Install dependencies

Run this in Google Colab with a GPU runtime.

In [ ]:
!pip install -q transformers accelerate peft trl bitsandbytes datasets scipy

## 3. Hugging Face authentication and checks

Enter `HF_TOKEN` and verify Hugging Face authentication. The setup cell above clones the public GitHub repository when needed, so `src/`, `configs/`, and `data/processed/en/` should already be available. Make sure access to `meta-llama/Meta-Llama-3-8B-Instruct` is approved for this token.

In [ ]:
import getpass
import os
from pathlib import Path

assert Path('src').exists(), 'Run %cd to the project root before continuing.'
assert Path('data/processed/en/eval_payload.jsonl').exists(), 'EN eval payload is missing.'

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')

!huggingface-cli whoami

## 4. Baseline generation

In [ ]:
!PYTHONPATH=$PWD python -m src.generation.generate_baseline \
  --language en \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --output-csv outputs/en/baseline_generations.csv \
  --max-samples 100

## 5. QLoRA training

In [ ]:
!PYTHONPATH=$PWD python -m src.training.train_qlora \
  --train-jsonl data/processed/en/train_payload.jsonl \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --adapter-dir adapters/en_lora_adapter \
  --max-train-samples 500 \
  --max-eval-samples 100

## 6. Fine-tuned generation

In [ ]:
!PYTHONPATH=$PWD python -m src.generation.generate_with_adapter \
  --language en \
  --eval-jsonl data/processed/en/eval_payload.jsonl \
  --adapter-dir adapters/en_lora_adapter \
  --output-csv outputs/en/finetuned_generations.csv \
  --max-samples 100

## 7. Stylometry and statistics

In [ ]:
!PYTHONPATH=$PWD python -m src.evaluation.run_stylometry \
  --language en \
  --human-jsonl data/processed/en/eval_payload.jsonl \
  --baseline-csv outputs/en/baseline_generations.csv \
  --finetuned-csv outputs/en/finetuned_generations.csv \
  --features-csv outputs/en/stylometric_features.csv \
  --summary-csv outputs/en/metric_summary.csv